In [2]:
pip install Prophet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 15.4 MB/s  0:00:00.6 MB/s eta 0:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 14.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [Prophet]━━━ 3/4 [Prophet]]
Note: you may need to restart the kernel to use updated packages.


In [3]:
from pathlib import Path
from datetime import datetime, timedelta, timezone

import pandas as pd
from coinbase.rest import RESTClient
from dotenv import load_dotenv
import os


# ---------------------------------------------------------
# Load environment variables
# ---------------------------------------------------------
BASE_DIR = Path(__file__).resolve().parent.parent
load_dotenv(BASE_DIR / ".env", override=True)

API_KEY = os.getenv("COINBASE_API_KEY")
API_SECRET = os.getenv("COINBASE_API_SECRET")


# ---------------------------------------------------------
# Coinbase client
# ---------------------------------------------------------

client = RESTClient(
    api_key=API_KEY,
    api_secret=API_SECRET
)


# ---------------------------------------------------------
# Get BTC historical candles
# ---------------------------------------------------------

def get_btc_candles(days=3):
    """
    Retrieve BTC-USD 30-minute candles from Coinbase.

    Parameters
    ----------
    days : int
        Number of days of historical data.

    Returns
    -------
    pandas.DataFrame
        BTC OHLCV data.
    """

    end = datetime.now(timezone.utc)
    start = end - timedelta(days=days)

    response = client.get_candles(
        product_id="BTC-USD",
        start=int(start.timestamp()),
        end=int(end.timestamp()),
        granularity="THIRTY_MINUTE"
    )

    candles = response["candles"]

    data = []

    for candle in candles:
        data.append({
            "timestamp": pd.to_datetime(
                int(candle["start"]),
                unit="s",
                utc=True
            ),
            "open": float(candle["open"]),
            "high": float(candle["high"]),
            "low": float(candle["low"]),
            "close": float(candle["close"]),
            "volume": float(candle["volume"])
        })

    df = pd.DataFrame(data)

    if df.empty:
        raise ValueError("No candle data returned from Coinbase.")

    df = df.sort_values("timestamp")
    df = df.reset_index(drop=True)

    return df

from prophet import Prophet
import pandas as pd


def predict_btc_prices(df, years=3):
    """
    Forecast BTC-USD prices using 30-minute historical candles.

    Parameters
    ----------
    df : pandas.DataFrame
        Historical BTC 30-minute OHLCV data.
        Must contain:
            timestamp
            open
            high
            low
            close
            volume

    years : int
        Number of years to forecast.

    Returns
    -------
    pandas.DataFrame
        Future 30-minute BTC price predictions.
    """

    # ---------------------------------------------------------
    # Prepare data for Prophet
    # ---------------------------------------------------------

    model_data = df[["timestamp", "close"]].copy()

    model_data = model_data.rename(
        columns={
            "timestamp": "ds",
            "close": "y"
        }
    )

    # Prophet works best with timezone-naive timestamps
    model_data["ds"] = (
        pd.to_datetime(model_data["ds"], utc=True)
        .dt.tz_localize(None)
    )

    # Make sure data is sorted
    model_data = model_data.sort_values("ds")

    # Remove duplicate timestamps
    model_data = model_data.drop_duplicates(
        subset=["ds"]
    )

    # Remove missing values
    model_data = model_data.dropna(
        subset=["ds", "y"]
    )

    # ---------------------------------------------------------
    # Create forecasting model
    # ---------------------------------------------------------

    model = Prophet(
        daily_seasonality=True,
        weekly_seasonality=True,
        yearly_seasonality=True,
        interval_width=0.95
    )

    # ---------------------------------------------------------
    # Train model
    # ---------------------------------------------------------

    print("Training BTC forecasting model...")

    model.fit(model_data)

    # ---------------------------------------------------------
    # Calculate number of 30-minute periods
    # ---------------------------------------------------------

    periods_per_day = 48
    days_per_year = 365

    future_periods = (
        years
        * days_per_year
        * periods_per_day
    )

    print(
        f"Generating {future_periods:,} "
        f"future 30-minute predictions..."
    )

    # ---------------------------------------------------------
    # Create future 30-minute timestamps
    # ---------------------------------------------------------

    future = model.make_future_dataframe(
        periods=future_periods,
        freq="30min"
    )

    # ---------------------------------------------------------
    # Generate prediction
    # ---------------------------------------------------------

    forecast = model.predict(future)

    # ---------------------------------------------------------
    # Keep only future predictions
    # ---------------------------------------------------------

    last_historical_timestamp = model_data["ds"].max()

    forecast = forecast[
        forecast["ds"] > last_historical_timestamp
    ]

    # ---------------------------------------------------------
    # Select useful columns
    # ---------------------------------------------------------

    forecast = forecast[
        [
            "ds",
            "yhat",
            "yhat_lower",
            "yhat_upper"
        ]
    ]

    # Rename columns
    forecast = forecast.rename(
        columns={
            "ds": "timestamp",
            "yhat": "predicted_close",
            "yhat_lower": "lower_bound",
            "yhat_upper": "upper_bound"
        }
    )

    forecast = forecast.reset_index(drop=True)

    return forecast

# ---------------------------------------------------------
# Test
# ---------------------------------------------------------

if __name__ == "__main__":

    df = get_btc_candles(days=3)

    print("\nBTC Market Data")
    print("----------------")
    print(df.head())

    print("\nData shape:")
    print(df.shape)

    print("\nLatest candle:")
    print(df.tail(1))

    df.to_csv("../data/btc_candles.csv", index=False)

    print("BTC data saved successfully.")

    forecast = predict_btc_prices(
        df,
        years=3
    )

    # Save predictions
    forecast.to_csv(
        "../data/btc_3year_forecast_30min.csv",
        index=False
    )

    print(
        "\n3-year 30-minute forecast saved successfully."
    )

    print("\nFirst predictions:")
    print(forecast.head())

    print("\nLast predictions:")
    print(forecast.tail())


BTC Market Data
----------------
                  timestamp      open      high       low     close  \
0 2026-09-09 16:30:00+00:00  78616.00  78901.04  78616.00  78783.18   
1 2026-09-09 17:00:00+00:00  78783.18  78916.64  78697.06  78839.62   
2 2026-09-09 17:30:00+00:00  78839.63  78848.47  78654.86  78754.89   
3 2026-09-09 18:00:00+00:00  78754.88  78774.72  78550.00  78614.75   
4 2026-09-09 18:30:00+00:00  78605.75  78634.57  78379.86  78399.35   

       volume  
0   79.890956  
1  116.123727  
2  101.259378  
3   80.712853  
4  122.122529  

Data shape:
(144, 6)

Latest candle:
                    timestamp      open     high      low     close    volume
143 2026-09-12 16:00:00+00:00  77363.97  77403.7  77321.8  77357.82  18.76923
BTC data saved successfully.


Yearly seasonality is enabled with less than 730 days (approximately 2 years) of history. The model may be under-identified, and the trend/seasonality decomposition can be unstable and dependent on the Prophet/Stan version. Consider disabling yearly seasonality or providing more history.
12:17:07 - cmdstanpy - INFO - Chain [1] start processing


Training BTC forecasting model...


12:17:08 - cmdstanpy - INFO - Chain [1] done processing
/opt/anaconda3/lib/python3.13/site-packages/prophet/forecaster.py:1660: RuntimeWarning: divide by zero encountered in matmul
  comp = np.matmul(X, beta_c.transpose())
/opt/anaconda3/lib/python3.13/site-packages/prophet/forecaster.py:1660: RuntimeWarning: overflow encountered in matmul
  comp = np.matmul(X, beta_c.transpose())
/opt/anaconda3/lib/python3.13/site-packages/prophet/forecaster.py:1660: RuntimeWarning: invalid value encountered in matmul
  comp = np.matmul(X, beta_c.transpose())
/opt/anaconda3/lib/python3.13/site-packages/prophet/forecaster.py:1852: RuntimeWarning: divide by zero encountered in matmul
  Xb_a = np.matmul(seasonal_features.values,
/opt/anaconda3/lib/python3.13/site-packages/prophet/forecaster.py:1852: RuntimeWarning: overflow encountered in matmul
  Xb_a = np.matmul(seasonal_features.values,
/opt/anaconda3/lib/python3.13/site-packages/prophet/forecaster.py:1852: RuntimeWarning: invalid value encountered in

Generating 52,560 future 30-minute predictions...

3-year 30-minute forecast saved successfully.

First predictions:
            timestamp  predicted_close   lower_bound   upper_bound
0 2026-09-12 16:30:00     77307.137702  76883.623224  77682.708750
1 2026-09-12 17:00:00     77250.361655  76879.404214  77632.423600
2 2026-09-12 17:30:00     77205.683959  76806.428091  77605.355049
3 2026-09-12 18:00:00     77177.224288  76735.947933  77590.160110
4 2026-09-12 18:30:00     77164.189726  76712.330033  77639.211963

Last predictions:
                timestamp  predicted_close   lower_bound   upper_bound
52555 2029-09-11 14:00:00    -1.229134e+07 -2.410164e+08  2.162583e+08
52556 2029-09-11 14:30:00    -1.229233e+07 -2.410251e+08  2.162640e+08
52557 2029-09-11 15:00:00    -1.229335e+07 -2.410345e+08  2.162703e+08
52558 2029-09-11 15:30:00    -1.229441e+07 -2.410436e+08  2.162761e+08
52559 2029-09-11 16:00:00    -1.229548e+07 -2.410528e+08  2.162817e+08
